# Specifying Model

In [2]:
from libpysal import weights
import esda
import numpy as np 
import matplotlib.pyplot as plt
import libpysal
from scipy import stats
from spreg import ML_Lag
import numpy as np
import libpysal
from spreg import ML_Lag, ML_Error
from scipy.stats import chi2
import statsmodels.api as sm
import numpy as np
import pandas as pd
import geopandas as gpd
import spreg

/hpc/m3/python/3.11.11/data_science-2025.08.21/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [3]:
from docx import Document

In [4]:
# setting wd 
import os
os.chdir('/users/bkung/fooddesertproject')

## Attempts to test specifications

In [ ]:
# # loading in data 
# data = gpd.read_file("modeling_data/modeling_data_final.gpkg")

In [ ]:
# # dropping NaNs 
# variables_to_use = ['distance_to_nearest_uf', 
#                  'distance_to_nearest_grocery',
#                  'distance_to_nearest_fm',
#                  'median_income',
#                  'pct_no_vehicle',
#                  'median_age',
#                  'walking_ind',
#                  'E_CHD'
#                 ]

# data_clean = data.dropna(subset=variables_to_use).copy()

# # logging income 
# data_clean['log_income'] = np.log(data_clean['median_income'])

In [ ]:
# # generating spatial weights 
# w = weights.distance.KNN.from_dataframe(data_clean, k=8)
# # row-standardization
# w.transform = "R"

In [ ]:
# # excluding bexar
# data_nobexar = data_clean[data_clean['COUNTY'] != 'Bexar County']

# # generating spatial weights 
# w = weights.Queen.from_dataframe(data_nobexar)
# # row-standardization
# w.transform = "R"

In [ ]:
# # Running OLS Model 
# ind_variables = ['distance_to_nearest_uf', 
#                  'distance_to_nearest_grocery',
#                  'distance_to_nearest_fm',
#                  'log_income',
#                  'pct_no_vehicle',
#                  'median_age',
#                  'walking_ind'
#                 ]

# ols = spreg.OLS(
#     # Dependent variable
#     data_[["E_DIABETES"]].values,
#     # Independent variables
#     data_nobexar[ind_variables].values,
#     # Dependent variable name
#     name_y="pct_diabetes",
#     # Independent variable name
#     name_x=ind_variables,
# )
# print(ols.summary)

# # generating moran's residuals 
# moran_residuals = spreg.MoranRes(ols, w, z=True)
# print(f"Observed Moran's I:  {moran_residuals.I:.4f}")
# print(f"Expected Moran's I:  {moran_residuals.eI:.4f}")
# print(f"Standardized Z-score: {moran_residuals.zI:.4f}")
# print(f"Analytical P-value:  {moran_residuals.p_norm:.4f}")

# # generating LM tests 
# lm_results = spreg.LMtests(ols, w)
# print("LM Error p-value:", lm_results.lme[1])
# print("LM Lag p-value:", lm_results.lml[1])

In [ ]:
# # Running OLS Model 
# ind_variables = ['distance_to_nearest_uf', 
#                  'distance_to_nearest_grocery',
#                  'distance_to_nearest_fm',
#                  'log_income',
#                  'pct_no_vehicle',
#                  'median_age',
#                  'walking_ind'
#                 ]

# ols = spreg.OLS(
#     # Dependent variable
#     data_clean[["E_DIABETES"]].values,
#     # Independent variables
#     data_clean[ind_variables].values,
#     # Dependent variable name
#     name_y="pct_diabetes",
#     # Independent variable name
#     name_x=ind_variables,
# )
# print(ols.summary)

# # generating moran's residuals 
# moran_residuals = spreg.MoranRes(ols, w, z=True)
# print(f"Observed Moran's I:  {moran_residuals.I:.4f}")
# print(f"Expected Moran's I:  {moran_residuals.eI:.4f}")
# print(f"Standardized Z-score: {moran_residuals.zI:.4f}")
# print(f"Analytical P-value:  {moran_residuals.p_norm:.4f}")

# # generating LM tests 
# lm_results = spreg.LMtests(ols, w)
# print("LM Error p-value:", lm_results.lme[1])
# print("LM Lag p-value:", lm_results.lml[1])

Given that the LM tests for the combined dataset indicate no spatial autocorrelation despite previously finding strong evidence for spatial autocorrelation in Dallas, we have chosen to run the models with counties separated. 

## Diagnostics 

In [5]:
# loading in data 
def read_county_data(county_name):
    gdf = gpd.read_file(f"extra_figures/modeling_data/{county_name}_modeling_final.gpkg")
    return gdf 

harris_data = read_county_data("harris")
bexar_data = read_county_data("bexar")
travis_data = read_county_data("travis")
dallas_data = read_county_data("dallas")
tarrant_data = read_county_data("tarrant")

In [6]:
# cleaning data (dropping NaNs)
def prepare_data(df): 
    variables_to_use = ['distance_to_nearest_uf', 
                     'distance_to_nearest_grocery',
                     'distance_to_nearest_fm',
                     'median_income',
                     'pct_no_vehicle',
                     'median_age',
                     'walking_ind',
                     'E_DIABETES'
                    ]
    df_clean = df.copy()
    df_clean = df_clean.dropna(subset=variables_to_use) 
    df_clean['log_income'] = np.log(df_clean['median_income'])
    rows_dropped = len(df) - len(df_clean)
    print(f"Dropped {rows_dropped} observations.") 
    print(f"Remaining observations: {len(df)}")
    return df_clean

harris_clean = prepare_data(harris_data)
bexar_clean = prepare_data(bexar_data)
travis_clean = prepare_data(travis_data)
dallas_clean = prepare_data(dallas_data)
tarrant_clean = prepare_data(tarrant_data)

Dropped 18 observations.
Remaining observations: 1289
Dropped 6 observations.
Remaining observations: 389
Dropped 7 observations.
Remaining observations: 312
Dropped 10 observations.
Remaining observations: 669
Dropped 7 observations.
Remaining observations: 464


In [7]:
# converting distance from feet to miles 
distance_vars = ['distance_to_nearest_uf', 
                       'distance_to_nearest_grocery',
                       'distance_to_nearest_fm'
                      ]

bexar_clean[distance_vars] = bexar_clean[distance_vars] / 5280
harris_clean[distance_vars] = harris_clean[distance_vars] / 5280
travis_clean[distance_vars] = travis_clean[distance_vars] / 5280
dallas_clean[distance_vars] = dallas_clean[distance_vars] / 5280
tarrant_clean[distance_vars] = tarrant_clean[distance_vars] / 5280

In [8]:
# generating spatial weights matrices (Queen Contiguity) 
def generate_weights(df): 
    w = weights.Queen.from_dataframe(df)
    # row-standardization
    w.transform = "R"
    return w 

harris_w = generate_weights(harris_clean)
bexar_w = generate_weights(bexar_clean)
travis_w = generate_weights(travis_clean)
dallas_w = generate_weights(dallas_clean)
tarrant_w = generate_weights(tarrant_clean)

/tmp/ipykernel_1665766/405985040.py:3: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(df)


In [8]:
# Running Travis County OLS as a first test 
# Running OLS baseline models 
ind_variables = ['distance_to_nearest_uf', 
                 'distance_to_nearest_grocery',
                 'distance_to_nearest_fm',
                 'log_income',
                 'pct_no_vehicle',
                 'median_age',
                 'walking_ind'
                ]
def run_ols(df, w, title): 
    # fitting OLS
    ols_model= spreg.OLS(
        # setting depvar 
        df[["E_DIABETES"]].values,
        # setting indvars 
        df[ind_variables].values,
        # naming depvar 
        name_y="pct_diabetes",
        # naming indvars 
        name_x=ind_variables,
        name_ds=f"{title}"
    )
    print(ols_model.summary)
    
    # generating moran's residuals 
    moran_residuals = spreg.MoranRes(ols_model, w, z=True)
    print(f"Observed Moran's I:  {moran_residuals.I:.4f}")
    print(f"Analytical P-value:  {moran_residuals.p_norm:.4f}")
    
    # generating LM tests 
    lm_results = spreg.LMtests(ols_model, w)
    print("LM Error p-value:", lm_results.lme[1])
    print("LM Lag p-value:", lm_results.lml[1])

    return ols_model
# printing the model 
travis_ols = run_ols(travis_clean, travis_w, "travis_county")

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :travis_county
Weights matrix      :        None
Dependent Variable  :pct_diabetes                Number of Observations:         305
Mean dependent var  :      9.9816                Number of Variables   :           8
S.D. dependent var  :      3.0490                Degrees of Freedom    :         297
R-squared           :      0.2350
Adjusted R-squared  :      0.2170
Sum squared residual:      2161.9                F-statistic           :     13.0357
Sigma-square        :       7.279                Prob(F-statistic)     :    1.24e-14
S.E. of regression  :       2.698                Log likelihood        :    -731.437
Sigma-square ML     :       7.088                Akaike info criterion :    1478.874
S.E of regression ML:      2.6624                Schwarz criterion     :    1508.636

----------------

Both LM tests are significant. Run restricted Spatial Durbin Models with likelihood-ratio tests to test for SEM, SAR, or SDM. 

In [34]:
# Travis County SDM With LR tests 
# constructing y (depvar)
y_col = 'E_DIABETES'
y = travis_clean[y_col].values.reshape(-1, 1)

x_cols = ind_variables
X = travis_clean[x_cols].values

# fitting unrestricted SDM with loglikelihoods 
sdm = ML_Lag(y, X, travis_w, slx_lags=1)
ll_sdm = sdm.logll

# fitting restricted SDM (SAR) with loglikelihoods 
sar = ML_Lag(y, X, travis_w)
ll_sar = sar.logll

# fitting restricted SDM (SEM) with loglikelihoods
sem = ML_Error(y, X, travis_w)
ll_sem = sem.logll

# likelihood-ratio tests 
df = X.shape[1] 

# LR Test for H0: theta = 0 (SDM vs SAR)
lr_stat_sar = 2 * (ll_sdm - ll_sar)
p_val_sar = chi2.sf(lr_stat_sar, df)

# LR Test for H0: theta + rho*beta = 0 (SDM vs SEM)
lr_stat_sem = 2 * (ll_sdm - ll_sem)
p_val_sem = chi2.sf(lr_stat_sem, df)

print(f"LR Test (SDM vs SAR): Stat = {lr_stat_sar:.4f}, p-value = {p_val_sar:.4e}")
print(f"LR Test (SDM vs SEM): Stat = {lr_stat_sem:.4f}, p-value = {p_val_sem:.4e}")

ML_Lag
ML_Lag
ML_Error
LR Test (SDM vs SAR): Stat = 22.1196, p-value = 2.4226e-03
LR Test (SDM vs SEM): Stat = 12.9466, p-value = 7.3420e-02


/users/bkung/.local/lib/python3.11/site-packages/spreg/ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


Results suggest an SDM. Given that SDM will collapse into either the SAR or SEM, for the sake of model consistency, I will run an SDM for all counties. 

## Baseline OLS Models

In [12]:
# Running OLS baseline models 
ind_variables = ['distance_to_nearest_uf', 
                 'distance_to_nearest_grocery',
                 'distance_to_nearest_fm',
                 'log_income',
                 'pct_no_vehicle',
                 'median_age',
                 'walking_ind'
                ]
def run_ols(df, w, title): 
    # fitting OLS
    ols_model= spreg.OLS(
        # setting depvar 
        df[["E_DIABETES"]].values,
        # setting indvars 
        df[ind_variables].values,
        w=w,
        spat_diag = True,
        moran = True,
        # naming depvar 
        name_y="pct_diabetes",
        # naming indvars 
        name_x=ind_variables,
        name_ds=f"{title}"
    )
    print(ols_model.summary)
    
    # generating moran's residuals 
    moran_residuals = spreg.MoranRes(ols_model, w, z=True)
    print(f"Observed Moran's I:  {moran_residuals.I:.4f}")
    print(f"Analytical P-value:  {moran_residuals.p_norm:.4f}")
    
    # generating LM tests 
    lm_results = spreg.LMtests(ols_model, w)
    print("LM Error p-value:", lm_results.lme[1])
    print("LM Lag p-value:", lm_results.lml[1])

    return ols_model
    
harris_ols= run_ols(harris_clean, harris_w, "harris_county")
bexar_ols = run_ols(bexar_clean, bexar_w, "bexar_county")
travis_ols = run_ols(travis_clean, travis_w, "travis_county")
dallas_ols = run_ols(dallas_clean, dallas_w, "dallas_county")
tarrant_ols = run_ols(tarrant_clean, tarrant_w, "tarrant_county")

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :harris_county
Weights matrix      :     unknown
Dependent Variable  :pct_diabetes                Number of Observations:        1271
Mean dependent var  :     14.2347                Number of Variables   :           8
S.D. dependent var  :      4.2284                Degrees of Freedom    :        1263
R-squared           :      0.5355
Adjusted R-squared  :      0.5330
Sum squared residual:     10546.1                F-statistic           :    208.0473
Sigma-square        :       8.350                Prob(F-statistic)     :  3.015e-205
S.E. of regression  :       2.890                Log likelihood        :   -3148.161
Sigma-square ML     :       8.298                Akaike info criterion :    6312.321
S.E of regression ML:      2.8805                Schwarz criterion     :    6353.502

----------------

In [13]:
# creating dictionary of all models 
models_dictionary = {
    "harris_ols": harris_ols,
    "bexar_ols":bexar_ols, 
    "travis_ols": travis_ols,
    "dallas_ols": dallas_ols,
    "tarrant_ols": tarrant_ols,
}

# # Open a clean text file and dump every model summary into it
# with open("Stat_Modeling/running_model/all_ols_models.txt", "w") as file:
#     for model_name, model_obj in models_dictionary.items():
#         file.write("=" * 60 + "\n")
#         file.write(f" SECTION: {model_name}\n")
#         file.write("=" * 60 + "\n\n")

#         # Write the entire raw spreg text output out to the file
#         file.write(model_obj.summary)
#         file.write("\n\n" + "_" * 60 + "\n\n")

# print("Saved!")
# Dictionaries to store separate diagnostic objects in memory
moran_results = {}
lm_test_results = {}

with open("final_figures/Table_3/all_ols_models.txt", "w") as file:
    for model_name, model_obj in models_dictionary.items():
        # --- 1. Extract and save separate Python objects ---
        
        # Moran's I tuple: (I_stat, z_stat, p_val)
        moran_obj = getattr(model_obj, "moran_res", None)
        moran_results[model_name] = moran_obj

        # Dictionary of LM test tuples: {test_name: (stat, p_val)}
        lm_obj = {
            "LM Lag": getattr(model_obj, "lm_lag", None),
            "LM Error": getattr(model_obj, "lm_error", None),
            "Robust LM Lag": getattr(model_obj, "rlm_lag", None),
            "Robust LM Error": getattr(model_obj, "rlm_error", None),
            "LM SARMA": getattr(model_obj, "lm_sarma", None),
        }
        lm_test_results[model_name] = lm_obj

        # --- 2. Write structured output to file ---
        
        file.write("=" * 60 + "\n")
        file.write(f" SECTION: {model_name}\n")
        file.write("=" * 60 + "\n\n")

        # Raw OLS summary
        file.write("--- OLS SUMMARY ---\n")
        file.write(model_obj.summary)
        file.write("\n\n")

        # Moran's I Residuals
        file.write("--- MORAN'S I ON RESIDUALS ---\n")
        if moran_obj is not None:
            file.write(f"Moran's I Stat : {moran_obj[0]:.4f}\n")
            file.write(f"z-value        : {moran_obj[1]:.4f}\n")
            file.write(f"p-value        : {moran_obj[2]:.4f}\n\n")
        else:
            file.write("Moran's I residuals not available.\n\n")

        # LM Tests
        file.write("--- LAGRANGE MULTIPLIER (LM) TESTS ---\n")
        for test_name, test_tuple in lm_obj.items():
            if test_tuple is not None:
                file.write(f"{test_name:<18}: Stat = {test_tuple[0]:.4f}, p-value = {test_tuple[1]:.4f}\n")
            else:
                file.write(f"{test_name:<18}: N/A\n")

        file.write("\n" + "_" * 60 + "\n\n")

## Spatial Durbin Models

In [ ]:
# Running Spatial Durbin Models 

def run_sdm(df, w, title): 
    # constructing matrix y (dependent variable)
    y = df[["E_DIABETES"]].values.reshape(-1, 1)
    # constructing matrix X (independent variables) 
    X = df[ind_variables].values
    # fitting SDM via maximum likelihood 
    sdm_model = ML_Lag(
        y, 
        X, 
        w, 
        slx_lags=1,
        name_y='pct_diabetes', 
        name_x=ind_variables,
        spat_impacts = "simple",
        name_w='queen_contig',
        name_ds=title
    )
    print(sdm_model.summary)
    return sdm_model

harris_sdm = run_sdm(harris_clean, harris_w, "harris_county")
bexar_sdm = run_sdm(bexar_clean, bexar_w, "bexar_county")
travis_sdm = run_sdm(travis_clean, travis_w, "travis_county")
dallas_sdm = run_sdm(dallas_clean, dallas_w, "dallas_county")
tarrant_sdm = run_sdm(tarrant_clean, tarrant_w, "tarrant_county")

In [15]:
# Create a dictionary of all your fitted models
models_dictionary = {
    "harris_sdm": harris_sdm,
    "dallas_sdm": dallas_sdm,
    "tarrant_sdm": tarrant_sdm,
    "bexar_sdm": bexar_sdm,
    "travis_sdm": travis_sdm
}

# Open a clean text file and dump every model summary into it
with open("Stat_Modeling/running_model/all_spatial_durbin_models.txt", "w") as file:
    for model_name, model_obj in models_dictionary.items():
        file.write("=" * 60 + "\n")
        file.write(f" SECTION: {model_name}\n")
        file.write("=" * 60 + "\n\n")

        # Write the entire raw spreg text output out to the file
        file.write(model_obj.summary)
        file.write("\n\n" + "_" * 60 + "\n\n")

print("Saved!")

Saved!


In [16]:
doc = Document()
doc.add_heading("All Spatial Durbin Models", level=1)

for model_name, model_obj in models_dictionary.items():
    doc.add_heading(model_name, level=2)

    # Add the text as a code-block style paragraph so everything aligns perfectly
    p = doc.add_paragraph()
    p.paragraph_format.space_after = (
        12  # Add clean spacing after the block
    )

    # Use a monospaced font (like Consolas) so the text tables stay straight
    run = p.add_run(model_obj.summary)
    run.font.name = "Consolas"

doc.save("Stat_Modeling/running_model/spatial_durbin_models_all.docx")
print("Saved!")

Saved!
